# 🚀 SERVE YOUR TRAINED MODEL FROM GOOGLE DRIVE
Run this in Colab to serve checkpoint-best via ngrok

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Your checkpoint path
CHECKPOINT_PATH = '/content/drive/MyDrive/TEKNOFEST_2025_Training_Runs/20250815_022938/checkpoints/checkpoint-best'
print(f'Loading model from: {CHECKPOINT_PATH}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading model from: /content/drive/MyDrive/TEKNOFEST_2025_Training_Runs/20250815_022938/checkpoints/checkpoint-best


In [32]:
!pip install --upgrade --no-cache-dir --no-deps unsloth_zoo -q
!pip install --upgrade --force-reinstall --no-deps torchvision -q
!pip install flask flask-cors pyngrok -q

In [36]:
# Cell 2: Load Model (restart runtime first if needed)
def load_model():
    import os

    # Your checkpoint path
    CHECKPOINT_PATH = '/content/drive/MyDrive/TEKNOFEST_2025_Training_Runs/20250815_022938/checkpoints/checkpoint-best'

    # Option 1: Try with unsloth
    try:
        import unsloth
        from unsloth import FastLanguageModel

        print("Loading with Unsloth...")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name="unsloth/gemma-3n-E4B-it",
            max_seq_length=2048,
            dtype=None,
            load_in_4bit=True,
            device_map="auto"
        )

        # Try to load your adapter
        if os.path.exists(CHECKPOINT_PATH):
            try:
                from peft import PeftModel
                if os.path.exists(os.path.join(CHECKPOINT_PATH, 'adapter_config.json')):
                    model = PeftModel.from_pretrained(model, CHECKPOINT_PATH)
                    print("✅ Your LoRA adapter loaded!")
                else:
                    print("⚠️ Using base model (no adapter found)")
            except Exception as e:
                print(f"⚠️ Adapter loading failed: {e}")

        FastLanguageModel.for_inference(model)
        print("✅ Unsloth model ready")
        return model, tokenizer, "unsloth"

    except Exception as e:
        print(f"Unsloth failed: {e}")

        # Option 2: Fallback to transformers
        try:
            from transformers import AutoModelForCausalLM, AutoTokenizer
            import torch

            print("Loading with transformers...")
            model = AutoModelForCausalLM.from_pretrained(
                "google/gemma-2-2b-it",
                device_map="auto",
                torch_dtype=torch.float16
            )

            tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b-it")
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token

            model.eval()
            print("✅ Transformers model ready")
            return model, tokenizer, "transformers"

        except Exception as e2:
            print(f"All loading methods failed: {e2}")
            raise e2

In [37]:
# Cell 3: Create Server
def create_server(model, tokenizer, model_type):
    from flask import Flask, request, jsonify
    from flask_cors import CORS
    import re

    app = Flask(__name__)
    CORS(app)

    # System prompt for Turkish telco
    SYSTEM_PROMPT = """Sen Türk telekom müşteri hizmetleri asistanısın.
Müşterilerin duygularına uygun yanıt ver ve gerekli araçları [araç_adı] formatında kullan.

Kullanabileceğin araçlar:
[get_current_balance] - Bakiye sorgula
[check_data_usage] - İnternet kullanımı
[activate_esim] - eSIM aktivasyonu
[create_support_ticket] - Destek talebi

Duygulara göre yanıt ver:
- angry: Özür dile, hızlı çözüm sun
- neutral: Profesyonel ol"""

    @app.route('/health', methods=['GET'])
    def health():
        return jsonify({
            'status': 'healthy',
            'model_type': model_type,
            'ready': True
        })

    @app.route('/predict', methods=['POST'])
    def predict():
        try:
            data = request.json
            text = data.get('text', '')
            emotion = data.get('emotion', 'neutral')

            # Build prompt
            prompt = f"""{SYSTEM_PROMPT}

<start_of_turn>user
<emotion>{emotion}</emotion>
{text}
<end_of_turn>
<start_of_turn>assistant"""

            # Generate based on model type
            if model_type == "unsloth":
                # Unsloth method
                inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=200,
                    temperature=0.7,
                    do_sample=True,
                    top_p=0.95
                )
                response = tokenizer.decode(outputs[0], skip_special_tokens=True)

            else:
                # Transformers method
                inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=200,
                        temperature=0.7,
                        do_sample=True,
                        top_p=0.95,
                        pad_token_id=tokenizer.eos_token_id
                    )
                response = tokenizer.decode(outputs[0], skip_special_tokens=True)

            # Extract assistant response
            if 'assistant' in response:
                response = response.split('assistant')[-1].strip()

            # Extract tools
            tools_found = re.findall(r'\[([^\]]+)\]', response)

            return jsonify({
                'generated_text': response,
                'emotion_detected': emotion,
                'tools_extracted': tools_found,
                'model_type': model_type,
                'success': True
            })

        except Exception as e:
            return jsonify({'error': str(e), 'success': False}), 500

    return app

In [38]:
# Cell 4: Start Server with ngrok
def start_server(app):
    from pyngrok import ngrok
    import threading

    # Configure ngrok
    ngrok.set_auth_token("31Iyz0YNz20h4XPZAhCdzH9mQfa_7pS9XT3qX1N6YC3kS6tZY")

    # Kill existing tunnels
    ngrok.kill()

    # Start tunnel
    public_url = ngrok.connect(5000)

    print("="*60)
    print("🎉 MODEL SERVER READY!")
    print("="*60)
    print(f"Your model URL: {public_url}")
    print("="*60)
    print(f"\nUse this URL in your local system:")
    print(f"./RUN_COMPLETE_SYSTEM.sh {public_url}")
    print("\n⚠️ Keep this running!")
    print("="*60)

    # Start Flask server
    app.run(port=5000, debug=False, host='0.0.0.0')

In [40]:
# Cell 5: Complete Setup (run all at once)
def complete_setup():
    print("🚀 Starting complete setup...")

    # Step 1: Setup
    setup_colab()

    # Step 2: Load model
    model, tokenizer, model_type = load_model()

    # Step 3: Create server
    app = create_server(model, tokenizer, model_type)

    # Step 4: Start server
    start_server(app)

In [ ]:
#!/usr/bin/env python3
"""
TEKNOFEST 2025 - ALL-IN-ONE COLAB MODEL SERVER
Complete model server in single file - just run in Colab!
"""

import subprocess
import sys
import os

def install_requirements():
    """Install all required packages"""
    print("📦 Installing requirements...")

    requirements = [
        'flask', 'flask-cors', 'pyngrok', 'torch', 'transformers',
        'accelerate', 'bitsandbytes', 'peft'
    ]

    for req in requirements:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', req, '-q'])
        except:
            print(f"Failed to install {req}, continuing...")

    # Try to install unsloth
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'unsloth[colab-new]@git+https://github.com/unslothai/unsloth.git', '-q'])
        print("✅ Unsloth installed")
    except:
        print("⚠️ Unsloth installation failed, will use transformers")

def mount_drive():
    """Mount Google Drive"""
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("✅ Drive mounted")
        return True
    except:
        print("⚠️ Not in Colab environment")
        return False

def load_model():
    """Load model with multiple fallback methods"""

    # Define checkpoint path
    CHECKPOINT_PATH = '/content/drive/MyDrive/TEKNOFEST_2025_Training_Runs/20250815_022938/checkpoints/checkpoint-best'

    print("🤖 Loading model...")

    # Method 1: Try Unsloth
    try:
        print("Attempting Unsloth loading...")
        import unsloth
        from unsloth import FastLanguageModel
        import torch

        # Load base model
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name="unsloth/gemma-3n-E4B-it",
            max_seq_length=2048,
            dtype=None,
            load_in_4bit=True,
            device_map="auto"
        )

        # Try to load checkpoint adapter
        if os.path.exists(CHECKPOINT_PATH):
            try:
                from peft import PeftModel
                adapter_config = os.path.join(CHECKPOINT_PATH, 'adapter_config.json')
                if os.path.exists(adapter_config):
                    model = PeftModel.from_pretrained(model, CHECKPOINT_PATH)
                    print("✅ Your trained LoRA adapter loaded!")
                    model_status = "trained"
                else:
                    print("⚠️ No adapter found, using base model")
                    model_status = "base"
            except Exception as e:
                print(f"⚠️ Adapter loading failed: {e}")
                model_status = "base"
        else:
            print("⚠️ Checkpoint path not found, using base model")
            model_status = "base"

        # Enable inference mode
        FastLanguageModel.for_inference(model)

        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        print(f"✅ Unsloth model loaded ({model_status})")
        return model, tokenizer, "unsloth", model_status

    except Exception as e:
        print(f"Unsloth failed: {e}")

    # Method 2: Fallback to Transformers
    try:
        print("Attempting Transformers loading...")
        from transformers import AutoModelForCausalLM, AutoTokenizer
        import torch

        model = AutoModelForCausalLM.from_pretrained(
            "google/gemma-2-2b-it",
            device_map="auto",
            torch_dtype=torch.float16,
            trust_remote_code=True
        )

        tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b-it")

        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        model.eval()
        print("✅ Transformers model loaded (base)")
        return model, tokenizer, "transformers", "base"

    except Exception as e:
        print(f"Transformers failed: {e}")
        raise Exception("All model loading methods failed!")

def create_flask_app(model, tokenizer, model_type, model_status):
    """Create Flask application"""

    from flask import Flask, request, jsonify
    from flask_cors import CORS
    import re
    import torch

    app = Flask(__name__)
    CORS(app)

    # System prompt for Turkish telco
    SYSTEM_PROMPT = """Sen Türk telekom müşteri hizmetleri uzmanısın.
Müşterilerin duygularına uygun yanıt ver ve gerekli araçları [araç_adı] formatında kullan.

Kullanabileceğin araçlar:
[get_current_balance] - Bakiye kontrolü
[check_data_usage] - İnternet kullanım kontrolü
[view_current_plan] - Mevcut paket bilgisi
[activate_esim] - eSIM aktivasyonu
[create_support_ticket] - Destek talebi oluşturma
[troubleshoot_connection] - Bağlantı sorunları

Duygusal yanıtlar:
- angry: "Yaşadığınız sorun için özür dileriz, hemen yardımcı oluyorum."
- sad: "Size yardımcı olmak için buradayım."
- confused: "Size açıklayayım."
- happy: "Memnuniyetiniz bizim için önemli!"
- neutral: "Size nasıl yardımcı olabilirim?"

Yanıtlarında gerekli araçları [araç_adı] formatında belirt."""

    @app.route('/health', methods=['GET'])
    def health():
        return jsonify({
            'status': 'healthy',
            'model_type': model_type,
            'model_status': model_status,
            'checkpoint_loaded': model_status == "trained",
            'ready': True,
            'system': 'TEKNOFEST 2025 Model Server'
        })

    @app.route('/predict', methods=['POST'])
    def predict():
        try:
            data = request.json or {}
            text = data.get('text', '')
            emotion = data.get('emotion', 'neutral')

            if not text.strip():
                return jsonify({'error': 'Text is required', 'success': False}), 400

            # Build complete prompt
            prompt = f"""{SYSTEM_PROMPT}

<start_of_turn>user
<emotion>{emotion}</emotion>
{text}
<end_of_turn>
<start_of_turn>assistant"""

            # Generate response
            inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=200,
                    temperature=0.7,
                    do_sample=True,
                    top_p=0.95,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id
                )

            # Decode response
            full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

            # Extract assistant part
            if 'assistant' in full_response:
                response = full_response.split('assistant')[-1].strip()
            else:
                response = full_response.replace(prompt, '').strip()

            # Extract tools
            tools_found = re.findall(r'\[([^\]]+)\]', response)

            # Clean response
            response = response[:500]  # Limit length

            print(f"Generated: {response[:100]}...")

            return jsonify({
                'generated_text': response,
                'emotion_detected': emotion,
                'tools_extracted': tools_found,
                'model_type': model_type,
                'model_status': model_status,
                'success': True
            })

        except Exception as e:
            print(f"Prediction error: {e}")
            return jsonify({
                'error': str(e),
                'success': False,
                'model_type': model_type
            }), 500

    @app.route('/test', methods=['GET'])
    def test():
        # Simple test endpoint
        test_response = "Merhaba! TEKNOFEST 2025 model sunucusu çalışıyor. [get_current_balance] aracını kullanabilirim."
        tools = re.findall(r'\[([^\]]+)\]', test_response)

        return jsonify({
            'message': test_response,
            'tools_found': tools,
            'model_type': model_type,
            'model_status': model_status,
            'working': True
        })

    return app

def start_ngrok_server(app):
    """Start ngrok tunnel and Flask server"""

    from pyngrok import ngrok

    # Configure ngrok with your token
    ngrok.set_auth_token("31Iyz0YNz20h4XPZAhCdzH9mQfa_7pS9XT3qX1N6YC3kS6tZY")

    # Kill existing tunnels
    try:
        ngrok.kill()
    except:
        pass

    # Create tunnel
    public_url = ngrok.connect(5000)

    print("\n" + "="*70)
    print("🎉 TEKNOFEST 2025 MODEL SERVER READY!")
    print("="*70)
    print(f"📡 Public URL: {public_url}")
    print("="*70)
    print(f"🔗 Use this URL in your local system:")
    print(f"   ./RUN_COMPLETE_SYSTEM.sh {public_url}")
    print("="*70)
    print("📋 Available endpoints:")
    print(f"   {public_url}/health   - Health check")
    print(f"   {public_url}/predict  - AI predictions")
    print(f"   {public_url}/test     - Quick test")
    print("="*70)
    print("⚠️  KEEP THIS CELL RUNNING!")
    print("="*70)

    # Start Flask server
    print("🚀 Starting Flask server...")
    app.run(host='0.0.0.0', port=5000, debug=False)

def main():
    """Main execution function"""

    print("""
╔════════════════════════════════════════════════════════════╗
║              TEKNOFEST 2025 - MODEL SERVER                ║
║                All-in-One Colab Setup                     ║
╚════════════════════════════════════════════════════════════╝
    """)

    try:


        # Step 2: Mount drive
        mount_drive()

        # Step 3: Load model
        model, tokenizer, model_type, model_status = load_model()

        # Step 4: Create Flask app
        app = create_flask_app(model, tokenizer, model_type, model_status)

        # Step 5: Start server with ngrok
        start_ngrok_server(app)

    except KeyboardInterrupt:
        print("\n⏹️  Server stopped by user")
    except Exception as e:
        print(f"\n❌ Setup failed: {e}")
        print("\nTroubleshooting:")
        print("1. Restart runtime and try again")
        print("2. Check if your checkpoint path exists")
        print("3. Ensure you have GPU runtime selected")

if __name__ == "__main__":
    main()


╔════════════════════════════════════════════════════════════╗
║              TEKNOFEST 2025 - MODEL SERVER                ║  
║                All-in-One Colab Setup                     ║
╚════════════════════════════════════════════════════════════╝
    
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted
🤖 Loading model...
Attempting Unsloth loading...


/tmp/ipython-input-2527751040.py:55: UserWarning: WARNING: Unsloth should be imported before trl, transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth


Unsloth: We'll be using `/tmp/unsloth_compiled_cache` for temporary Unsloth patches.
Standard import failed for UnslothSFTTrainer: cannot access local variable 'old_path' where it is not associated with a value. Using tempfile instead!
==((====))==  Unsloth 2025.8.5: Fast Gemma3N patching. Transformers: 4.55.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gemma3N does not support SDPA - switching to eager!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.72G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/peft/peft_model.py:585: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.language_model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.language_model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.language_model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.language_model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.language_model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.language_model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.language_model.layers.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.language_model.layers.0.mlp.gate_proj.lora_

✅ Your trained LoRA adapter loaded!
✅ Unsloth model loaded (trained)

🎉 TEKNOFEST 2025 MODEL SERVER READY!
📡 Public URL: NgrokTunnel: "https://7983b23bdd57.ngrok-free.app" -> "http://localhost:5000"
🔗 Use this URL in your local system:
   ./RUN_COMPLETE_SYSTEM.sh NgrokTunnel: "https://7983b23bdd57.ngrok-free.app" -> "http://localhost:5000"
📋 Available endpoints:
   NgrokTunnel: "https://7983b23bdd57.ngrok-free.app" -> "http://localhost:5000"/health   - Health check
   NgrokTunnel: "https://7983b23bdd57.ngrok-free.app" -> "http://localhost:5000"/predict  - AI predictions
   NgrokTunnel: "https://7983b23bdd57.ngrok-free.app" -> "http://localhost:5000"/test     - Quick test
⚠️  KEEP THIS CELL RUNNING!
🚀 Starting Flask server...
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [15/Aug/2025 03:48:04] "GET /health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Aug/2025 03:49:16] "GET /health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Aug/2025 03:51:18] "GET /health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Aug/2025 03:51:31] "POST /predict HTTP/1.1" 400 -
INFO:werkzeug:127.0.0.1 - - [15/Aug/2025 03:52:17] "GET /health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Aug/2025 03:52:29] "POST /predict HTTP/1.1" 400 -
INFO:werkzeug:127.0.0.1 - - [15/Aug/2025 03:54:19] "POST /predict HTTP/1.1" 400 -


In [ ]:
# Run the server (THIS WILL BLOCK - Keep running!)
print('🚀 Starting model server...')
print('DO NOT STOP THIS CELL!')
app.run(port=5000, debug=False)

## 🧪 Test Your Model (Optional)
Run this in a separate cell to test

In [ ]:
# Test the model directly
test_prompt = '''<start_of_turn>user
<emotion>angry</emotion>
Faturamı öğrenmek istiyorum!
<end_of_turn>
<start_of_turn>assistant'''

inputs = tokenizer(test_prompt, return_tensors='pt').to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
        top_p=0.95
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print('Model response:')
print(response.split('assistant')[-1].strip())

In [23]:
!pip uninstall -y torchvision -q
!pip install torchvision -q
print('✅ torchvision reinstalled')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xformers 0.0.29.post3 requires torch==2.6.0, but you have torch 2.8.0 which is incompatible.
fastai 2.7.19 requires torch<2.7,>=1.10, but you have torch 2.8.0 which is incompatible.
torchaudio 2.6.0+cu124 requires torch==2.6.0, but you have torch 2.8.0 which is incompatible.
✅ torchvision reinstalled
